In [1]:
# Cell 1: Import required libraries

import os
import re
import glob
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)


Saving medium_data.csv to medium_data.csv
Uploaded files:
medium_data.csv


In [3]:


import pandas as pd

df = pd.read_csv("medium_data.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)


Dataset shape: (6508, 10)

Column names:
['id', 'url', 'title', 'subtitle', 'image', 'claps', 'responses', 'reading_time', 'publication', 'date']

First 5 rows:


,id,url,title,subtitle,image,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/a-beginners-gui...,A Beginner’s Guide to Word Embedding with Gens...,NaN,1.png,850,8,8,Towards Data Science,2019-05-30
1,2,https://towardsdatascience.com/hands-on-graph-...,Hands-on Graph Neural Networks with PyTorch & ...,NaN,2.png,1100,11,9,Towards Data Science,2019-05-30
2,3,https://towardsdatascience.com/how-to-use-ggpl...,How to Use ggplot2 in Python,A Grammar of Graphics for Python,3.png,767,1,5,Towards Data Science,2019-05-30
3,4,https://towardsdatascience.com/databricks-how-...,Databricks: How to Save Files in CSV on Your L...,When I work on Python projects dealing…,4.jpeg,354,0,4,Towards Data Science,2019-05-30
4,5,https://towardsdatascience.com/a-step-by-step-...,A Step-by-Step Implementation of Gradient Desc...,One example of building neural…,5.jpeg,211,3,4,Towards Data Science,2019-05-30



Data types:
id               int64
url             object
title           object
subtitle        object
image           object
claps            int64
responses       object
reading_time     int64
publication     object
date            object
dtype: object


In [4]:

# STEP 3: Combine title and subtitle text

import pandas as pd

df = pd.read_csv("medium_data.csv")

# Replace missing text with empty strings
df["title"] = df["title"].fillna("").astype(str)
df["subtitle"] = df["subtitle"].fillna("").astype(str)

# Combine title and subtitle into one text field
df["combined_text"] = (
    df["title"].str.strip() + ". " +
    df["subtitle"].str.strip()
).str.strip()

# Remove rows where both title and subtitle are empty
df = df[df["combined_text"].str.len() > 0].copy()

print("Total usable articles:", len(df))

print("\nSample combined text:")
for text in df["combined_text"].head(5):
    print("-", text)


Total usable articles: 6508

Sample combined text:
- A Beginner’s Guide to Word Embedding with Gensim Word2Vec Model.
- Hands-on Graph Neural Networks with PyTorch & PyTorch Geometric.
- How to Use ggplot2 in Python. A Grammar of Graphics for Python
- Databricks: How to Save Files in CSV on Your Local Computer. When I work on Python projects dealing…
- A Step-by-Step Implementation of Gradient Descent and Backpropagation. One example of building neural…


In [5]:

# STEP 4: Split the combined text into sentences

import re

all_text = " ".join(df["combined_text"].tolist())

# Split text at sentence-ending punctuation
sentences = re.split(r'(?<=[.!?])\s+', all_text)

# Remove empty sentences and very short entries
sentences = [
    sentence.strip()
    for sentence in sentences
    if len(sentence.strip().split()) >= 3
]

print("Total sentences available:", len(sentences))

print("\nFirst 10 sentences:")
for sentence in sentences[:10]:
    print("-", sentence)


Total sentences available: 7323

First 10 sentences:
- A Beginner’s Guide to Word Embedding with Gensim Word2Vec Model.
- Hands-on Graph Neural Networks with PyTorch & PyTorch Geometric.
- How to Use ggplot2 in Python.
- A Grammar of Graphics for Python Databricks: How to Save Files in CSV on Your Local Computer.
- When I work on Python projects dealing… A Step-by-Step Implementation of Gradient Descent and Backpropagation.
- One example of building neural… An Easy Introduction to SQL for Data Scientists.
- Hypothesis testing visualized.
- Literally seeing how stat tests work Introduction to Latent Matrix Factorization Recommender Systems.
- Which 2020 Candidate is the Best at Twitter?.
- A Data Analysis of the 2020 Democratic Candidate Twitter Accounts What if AI model understanding were easy?.


In [6]:

# STEP 5: Clean and prepare the sentence corpus

import re

def clean_sentence(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s.!?'’-]", " ", text)
    return text.strip()

cleaned_sentences = [
    clean_sentence(sentence)
    for sentence in sentences
]

# Remove empty sentences and sentences with fewer than 3 words
cleaned_sentences = [
    sentence
    for sentence in cleaned_sentences
    if len(sentence.split()) >= 3
]

# Remove duplicate sentences
cleaned_sentences = list(dict.fromkeys(cleaned_sentences))

print("Sentences after cleaning:", len(cleaned_sentences))

print("\nFirst 10 cleaned sentences:")
for sentence in cleaned_sentences[:10]:
    print("-", sentence)


Sentences after cleaning: 7307

First 10 cleaned sentences:
- a beginner’s guide to word embedding with gensim word2vec model.
- hands-on graph neural networks with pytorch   pytorch geometric.
- how to use ggplot2 in python.
- a grammar of graphics for python databricks  how to save files in csv on your local computer.
- when i work on python projects dealing  a step-by-step implementation of gradient descent and backpropagation.
- one example of building neural  an easy introduction to sql for data scientists.
- hypothesis testing visualized.
- literally seeing how stat tests work introduction to latent matrix factorization recommender systems.
- which 2020 candidate is the best at twitter?.
- a data analysis of the 2020 democratic candidate twitter accounts what if ai model understanding were easy?.


In [7]:

# STEP 6: Split the corpus into training and testing sets

from sklearn.model_selection import train_test_split

train_sentences, test_sentences = train_test_split(
    cleaned_sentences,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Total sentences:", len(cleaned_sentences))
print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))

print("\nSample training sentences:")
for sentence in train_sentences[:5]:
    print("-", sentence)


Total sentences: 7307
Training sentences: 5845
Testing sentences: 1462

Sample training sentences:
- predicting incremental gains of  introduction to backtesting trading strategies.
- when my remote leadership skills took off.
- and why should you dear reader  care?
- islam is full of beautiful things you didn’t know before.
- from leaves to computer vision  an anthology of product packaging..


In [8]:

# STEP 7: Tokenize words and build the vocabulary

from tensorflow.keras.preprocessing.text import Tokenizer

MAX_VOCAB = 12000

tokenizer = Tokenizer(
    num_words=MAX_VOCAB,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_sentences)

vocab_size = min(
    MAX_VOCAB,
    len(tokenizer.word_index) + 1
)

print("Total unique words:", len(tokenizer.word_index))
print("Vocabulary size used by model:", vocab_size)

print("\nSample word-to-index mapping:")
for word, index in list(tokenizer.word_index.items())[:20]:
    print(f"{word} -> {index}")


Total unique words: 8705
Vocabulary size used by model: 8706

Sample word-to-index mapping:
<OOV> -> 1
the -> 2
to -> 3
a -> 4
and -> 5
of -> 6
strong -> 7
how -> 8
your -> 9
in -> 10
you -> 11
for -> 12
markup -> 13
is -> 14
with -> 15
data -> 16
i -> 17
what -> 18
class -> 19
on -> 20


In [9]:

# STEP 8: Generate input-target sequences

MAX_CONTEXT = 5

def generate_sequences(sentence_list, tokenizer):
    sequences = []

    encoded_sentences = tokenizer.texts_to_sequences(
        sentence_list
    )

    for encoded in encoded_sentences:
        for i in range(1, len(encoded)):
            # Previous words become the input
            input_sequence = encoded[:i]

            # Keep only the most recent 5 words
            input_sequence = input_sequence[-MAX_CONTEXT:]

            # The next word becomes the target
            target_word = encoded[i]

            sequences.append((input_sequence, target_word))

    return sequences


train_pairs = generate_sequences(train_sentences, tokenizer)
test_pairs = generate_sequences(test_sentences, tokenizer)

print("Training input-target pairs:", len(train_pairs))
print("Testing input-target pairs:", len(test_pairs))

print("\nFirst 5 training examples:")

for input_sequence, target_word in train_pairs[:5]:
    input_words = [
        tokenizer.index_word.get(word_id, "<OOV>")
        for word_id in input_sequence
    ]

    target_text = tokenizer.index_word.get(
        target_word, "<OOV>"
    )

    print("Input:", " ".join(input_words))
    print("Target:", target_text)
    print("---")


Training input-target pairs: 61060
Testing input-target pairs: 15282

First 5 training examples:
Input: predicting
Target: incremental
---
Input: predicting incremental
Target: gains
---
Input: predicting incremental gains
Target: of
---
Input: predicting incremental gains of
Target: introduction
---
Input: predicting incremental gains of introduction
Target: to
---


In [10]:

# STEP 9: Pad sequences and create NumPy arrays

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = [pair[0] for pair in train_pairs]
y_train = [pair[1] for pair in train_pairs]

X_test = [pair[0] for pair in test_pairs]
y_test = [pair[1] for pair in test_pairs]

X_train = pad_sequences(
    X_train,
    maxlen=MAX_CONTEXT,
    padding="pre",
    truncating="pre"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_CONTEXT,
    padding="pre",
    truncating="pre"
)

y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nExample padded input:", X_train[0])
print("Example target:", y_train[0])


X_train shape: (61060, 5)
y_train shape: (61060,)
X_test shape: (15282, 5)
y_test shape: (15282,)

Example padded input: [  0   0   0   0 309]
Example target: 1762


In [11]:

# STEP 10: Build the LSTM model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:

# Build the model with an explicit input shape
model.build(input_shape=(None, MAX_CONTEXT))

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 64)          │       557,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8706)           │       565,890 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,156,098 (4.41 MB)

 Trainable params: 1,156,098 (4.41 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.0372 - loss: 7.3058 - val_accuracy: 0.0360 - val_loss: 7.0485
Epoch 2/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0506 - loss: 6.8098 - val_accuracy: 0.0627 - val_loss: 7.0062
Epoch 3/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0685 - loss: 6.6650 - val_accuracy: 0.0704 - val_loss: 7.0098
Epoch 4/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0826 - loss: 6.5357 - val_accuracy: 0.0919 - val_loss: 6.9520
Epoch 5/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1014 - loss: 6.3789 - val_accuracy: 0.0986 - val_loss: 6.8905
Epoch 6/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1144 - loss: 6.2313 - val_accuracy: 0.1028 - val_loss: 6.8561
Epoch 7/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1202 - loss: 6.1033 - val_accuracy: 0.1060 - val_loss: 6.8313
Epoch 8/8
430/430 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1291 - loss: 5.9935 - val_accuracy: 0.1158 - v

In [16]:

import numpy as np

# Evaluate the model on unseen test data
test_loss, top1_accuracy = model.evaluate(
    X_test, y_test, verbose=1
)

# Get predicted probabilities for all test examples
predictions = model.predict(X_test, batch_size=128, verbose=1)

# Find the 5 most probable next words
top5_indices = np.argsort(predictions, axis=1)[:, -5:]

# Check whether the actual next word is in the top 5
top5_correct = np.any(
    top5_indices == y_test.reshape(-1, 1),
    axis=1
)

top5_accuracy = np.mean(top5_correct)

print(f"Test Loss: {test_loss:.4f}")
print(f"Top-1 Accuracy: {top1_accuracy * 100:.2f}%")
print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")


478/478 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.1201 - loss: 6.7662
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Test Loss: 6.7662
Top-1 Accuracy: 12.01%
Top-5 Accuracy: 24.16%


In [17]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_words(phrase, top_k=5):
    # Convert the input phrase into word IDs
    sequence = tokenizer.texts_to_sequences([phrase.lower()])[0]

    # Keep only the latest MAX_CONTEXT words
    sequence = sequence[-MAX_CONTEXT:]

    # Pad the input to the model's expected length
    padded = pad_sequences(
        [sequence],
        maxlen=MAX_CONTEXT,
        padding="pre",
        truncating="pre"
    )

    # Predict probabilities for the next word
    probabilities = model.predict(padded, verbose=0)[0]

    # Get the top-k word IDs
    top_ids = np.argsort(probabilities)[-top_k:][::-1]

    # Convert word IDs back to words
    index_word = tokenizer.index_word
    results = []

    for word_id in top_ids:
        word = index_word.get(int(word_id), "<unknown>")
        results.append((word, float(probabilities[word_id])))

    return results


test_phrases = [
    "a beginner's guide to",
    "graph neural networks with",
    "how to use",
    "machine learning",
    "data science"
]

for phrase in test_phrases:
    print(f"\nInput phrase: {phrase}")
    for word, probability in predict_next_words(phrase):
        print(f"  {word}: {probability:.4f}")



Input phrase: a beginner's guide to
  be: 0.0406
  the: 0.0317
  make: 0.0230
  get: 0.0202
  create: 0.0160

Input phrase: graph neural networks with
  the: 0.0442
  python: 0.0294
  a: 0.0230
  data: 0.0218
  your: 0.0144

Input phrase: how to use
  your: 0.0700
  the: 0.0499
  a: 0.0467
  and: 0.0165
  it: 0.0164

Input phrase: machine learning
  and: 0.0635
  is: 0.0584
  in: 0.0502
  for: 0.0310
  the: 0.0308

Input phrase: data science
  and: 0.0471
  in: 0.0364
  with: 0.0267
  for: 0.0227
  of: 0.0155


In [18]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def prepare_subset_data(sentences):
    sub_tokenizer = Tokenizer(
        num_words=MAX_VOCAB,
        oov_token="<OOV>"
    )
    sub_tokenizer.fit_on_texts(sentences)

    sub_vocab_size = min(
        MAX_VOCAB,
        len(sub_tokenizer.word_index) + 1
    )

    pairs = generate_sequences(sentences, sub_tokenizer)

    if not pairs:
        return None

    X = pad_sequences(
        [p[0] for p in pairs],
        maxlen=MAX_CONTEXT,
        padding="pre",
        truncating="pre"
    )
    y = np.array([p[1] for p in pairs], dtype=np.int32)

    return sub_tokenizer, sub_vocab_size, X, y


results = []

for fraction in [0.25, 0.50, 1.00]:
    subset_size = int(len(train_sentences) * fraction)
    subset = train_sentences[:subset_size]

    sub_tokenizer, sub_vocab_size, X_sub, y_sub = prepare_subset_data(subset)

    test_pairs_sub = generate_sequences(test_sentences, sub_tokenizer)
    X_test_sub = pad_sequences(
        [p[0] for p in test_pairs_sub],
        maxlen=MAX_CONTEXT,
        padding="pre",
        truncating="pre"
    )
    y_test_sub = np.array(
        [p[1] for p in test_pairs_sub],
        dtype=np.int32
    )

    sub_model = Sequential([
        Embedding(sub_vocab_size, 32),
        LSTM(32),
        Dropout(0.2),
        Dense(sub_vocab_size, activation="softmax")
    ])

    sub_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    sub_model.fit(
        X_sub,
        y_sub,
        epochs=3,
        batch_size=128,
        validation_split=0.1,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=1,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    test_loss, test_top1 = sub_model.evaluate(
        X_test_sub, y_test_sub, verbose=0
    )

    probs = sub_model.predict(X_test_sub, batch_size=128, verbose=0)
    top5_ids = np.argsort(probs, axis=1)[:, -5:]
    test_top5 = np.mean(
        np.any(top5_ids == y_test_sub[:, None], axis=1)
    )

    results.append({
        "Training data": f"{int(fraction * 100)}%",
        "Sentences": subset_size,
        "Top-1 accuracy (%)": round(test_top1 * 100, 2),
        "Top-5 accuracy (%)": round(test_top5 * 100, 2)
    })

for row in results:
    print(row)


{'Training data': '25%', 'Sentences': 1461, 'Top-1 accuracy (%)': 3.87, 'Top-5 accuracy (%)': np.float64(13.69)}
{'Training data': '50%', 'Sentences': 2922, 'Top-1 accuracy (%)': 3.87, 'Top-5 accuracy (%)': np.float64(13.45)}
{'Training data': '100%', 'Sentences': 5845, 'Top-1 accuracy (%)': 3.87, 'Top-5 accuracy (%)': np.float64(13.45)}


In [19]:

readme_content = """
# Next-Word Prediction Using LSTM

## Project Overview
This project predicts the next word based on words already
typed by a user. It uses a text corpus, tokenization, padded
input sequences, and an LSTM neural network.

## Dataset
Source: Kaggle - Next Word Prediction
https://www.kaggle.com/datasets/hsankesara/next-word-prediction

File used: medium_data.csv

The dataset contains 6,508 article metadata rows. Since full
article bodies were not available in the downloaded CSV, titles
and subtitles were used to construct approximately 7,307
cleaned sentence-like text samples. This is a limitation of
the available corpus and does not meet the target of 10,000
full sentences.

## Methodology
1. Load and inspect the CSV dataset.
2. Combine available titles and subtitles.
3. Clean and tokenize text.
4. Build a vocabulary.
5. Generate input-target word sequences.
6. Pad input sequences to a fixed length.
7. Train an LSTM next-word prediction model.
8. Evaluate Top-1 and Top-5 accuracy.
9. Compare smaller and larger training subsets.

## Model Architecture
- Embedding layer: 64 dimensions
- LSTM layer: 64 units
- Dropout: 0.2
- Dense output layer: softmax
- Optimizer: Adam
- Loss: sparse categorical cross-entropy
- Maximum context length: 5 words

## Main Model Results
- Training accuracy: 12.91%
- Validation accuracy: 11.58%
- Test Top-1 accuracy: 12.01%
- Test Top-5 accuracy: 24.16%
- Test loss: 6.7662

## Training Data Size Experiment

| Training portion | Sentences | Top-1 accuracy | Top-5 accuracy |
|---|---:|---:|---:|
| 25% | 1,461 | 3.87% | 13.69% |
| 50% | 2,922 | 3.87% | 13.45% |
| 100% | 5,845 | 3.87% | 13.45% |

The subset experiments used smaller models and only three epochs.
No clear accuracy improvement was observed in this experiment.
The results should be interpreted in light of the limited text
corpus and the simplified training setup.

## Limitations
- The CSV contains article metadata, not full article text.
- Some samples may combine titles and subtitles imperfectly.
- The corpus is smaller than the target of 10,000 sentences.
- The model's prediction accuracy is limited.

## Technologies
- Python
- Pandas
- NumPy
- TensorFlow / Keras
- Scikit-learn
- Google Colab

## How to Run
Open the notebook in Google Colab, upload `medium_data.csv`,
and run the cells from top to bottom.

## Future Improvements
- Use a larger corpus of complete sentences.
- Improve sentence-boundary handling.
- Train for more epochs with suitable regularization.
- Compare LSTM with GRU or Transformer-based models.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content.strip() + "\n")

print("README.md created successfully!")


README.md created successfully!
